In [ ]:
import numpy as np 
import pandas as pd 
import xgboost as xgb 
import csv as csv
from xgboost import plot_importance
from matplotlib import pyplot
from sklearn.model_selection import cross_val_score,KFold
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV   
from scipy.stats import skew
from collections import OrderedDict
from sklearn.inspection import permutation_importance
import shap
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, KFold
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time
from sklearn.metrics import make_scorer
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
import joblib
from django.conf import settings 
import os
from pathlib import Path

In [ ]:

def load_model_with_joblib_and_get_features(filename):
    
    
    model = joblib.load(filename)
    
    feature_names = model.get_booster().feature_names
    return model, feature_names

In [ ]:

Tm_xgb_model, Tm_feature_names = load_model_with_joblib_and_get_features('/path/to/example/ionic_liquid/static/model/Tm_xgb_model.joblib')
lnA_xgb_model, lnA_feature_names = load_model_with_joblib_and_get_features('/path/to/example/ionic_liquid/static/model/lnA_xgb_model.joblib')
Ea_xgb_model, Ea_feature_names = load_model_with_joblib_and_get_features('/path/to/example/ionic_liquid/static/model/Ea_xgb_model.joblib')
conductivity_xgb_model, conductivity_feature_names = load_model_with_joblib_and_get_features('/path/to/example/ionic_liquid/static/model/conductivity_xgb_model.joblib')
ECW_xgb_model, ECW_feature_names = load_model_with_joblib_and_get_features('/path/to/example/ionic_liquid/static/model/ECW_xgb_model.joblib')

In [ ]:
def predict_with_xgboost(df, model, features, result_column_name='prediction_conduvtivity(S/m)'):
    
    
    missing_features = [feature for feature in features if feature not in df.columns]
    if missing_features:
        
        print("Public message removed for release.", missing_features)
        
        raise ValueError("Public message removed for release.")
    
    
    X = df[features]

    
    predictions = model.predict(X)

    
    df_with_predictions = df.copy()
    df_with_predictions[result_column_name] = predictions

    return df_with_predictions

In [ ]:

IL_df = pd.read_excel("Ionic_liquid_list_descriptor.xlsx")  
predict_df = predict_with_xgboost(IL_df, Tm_xgb_model, Tm_feature_names, result_column_name='Tm(K)')
predict_df = predict_with_xgboost(predict_df, lnA_xgb_model, lnA_feature_names, result_column_name='lnA')
predict_df = predict_with_xgboost(predict_df, Ea_xgb_model, Ea_feature_names, result_column_name='Ea(kJ/mol)')
predict_df = predict_with_xgboost(predict_df, conductivity_xgb_model, conductivity_feature_names, result_column_name='conductivity(S/m)')
predict_df = predict_with_xgboost(predict_df, ECW_xgb_model, ECW_feature_names, result_column_name='ECW(V)')
predict_df.to_excel("Ionic_liquid_list_descriptor.xlsx", index=False)

In [ ]:

def filter_columns(df):
    
    
    columns_to_keep = ['Name', 'IL_SMILES', 'Cation_SMILES', 'Anion_SMILES', 
                       'Cation_SMILES_type', 'Tm(K)', 'lnA', 'Ea(kJ/mol)', 
                       'conductivity(S/m)', 'ECW(V)']
    
    
    filtered_df = df[columns_to_keep]
    
    return filtered_df

In [ ]:
predict_df_filter_columns = filter_columns(predict_df)
predict_df_filter_columns.to_excel("Ionic_liquid_list_output.xlsx",index=None)

In [ ]:
predict_df_filter_columns